In [3]:
import  pandas as pd
import numpy  as np

In [10]:

data = pd.read_csv("../data/credit_risk_dataset.csv")
data_clean = data.copy()

In [11]:
# Renommer  les  colonnes  en français 


colonnes_fr = {
    'person_age': 'age',
    'person_income': 'revenu_annuel',
    'person_home_ownership': 'statut_logement',
    'person_emp_length': 'anciennete_emploi',
    'loan_intent': 'motif_pret',
    'loan_grade': 'niveau_de_risque',
    'loan_amnt': 'montant_pret',
    'loan_int_rate': 'taux_interet',
    'loan_status': 'statut_pret',  
    'loan_percent_income': 'ratio_pret_revenu',
    'cb_person_default_on_file': 'defaut_historique_credit',
    'cb_person_cred_hist_length': 'duree_historique_credit'
}


data = data.rename(columns=colonnes_fr)





In [12]:


def explorer(data):
    print("Exploration des données")
    print("Nombre de lignes et de colonnes :", data.shape)
    print("Types de données :")
    print(data.dtypes)
    print("\nValeurs manquantes :")
    print(data.isnull().sum())
    print("Pourcentage  des valeurs  manquantes par colonnes: ")
    print(data_clean.isnull().mean() * 100)
    
    print("Statistiques descriptives :")
    print(data.describe())
    print("\nCorrélations :")
    print(data.corr(numeric_only=True))

explorer(data)


Exploration des données
Nombre de lignes et de colonnes : (32581, 12)
Types de données :
age                           int64
revenu_annuel                 int64
statut_logement                 str
anciennete_emploi           float64
motif_pret                      str
niveau_de_risque                str
montant_pret                  int64
taux_interet                float64
statut_pret                   int64
ratio_pret_revenu           float64
defaut_historique_credit        str
duree_historique_credit       int64
dtype: object

Valeurs manquantes :
age                            0
revenu_annuel                  0
statut_logement                0
anciennete_emploi            895
motif_pret                     0
niveau_de_risque               0
montant_pret                   0
taux_interet                3116
statut_pret                    0
ratio_pret_revenu              0
defaut_historique_credit       0
duree_historique_credit        0
dtype: int64
Pourcentage  des valeurs  manquant

In [ ]:
# Nettoyage des données

# Suppresion des doublons
data_clean=data_clean.drop_duplicates()

#  Pourcentage de valeurs manquantes par colonne
data_clean.isnull().mean() * 100





age                         0.000000
revenu_annuel               0.000000
statut_logement             0.000000
anciennete_emploi           2.736303
motif_pret                  0.000000
niveau_de_risque            0.000000
montant_pret                0.000000
taux_interet                9.547754
statut_pret                 0.000000
ratio_pret_revenu           0.000000
defaut_historique_credit    0.000000
duree_historique_credit     0.000000
dtype: float64

In [14]:
import numpy as np
import pandas as pd


def afficher_regles_expert():

    print(
        "Voici les recommandations des experts data scientists / MLOps pour le"
        " traitement des valeurs manquantes :"
    )
    print("1. VARIABLES NUMÉRIQUES :")
    print("\n")
    print(
        "   1- Médiane (Recommandé) : Indispensable en cas de valeurs"
        " extrêmes"
    )
    print("     (outliers) pour ne pas déformer la distribution.")
    print("   2- Moyenne : À réserver aux distributions parfaitement normales.")
    print("   3-Supprimer : À éviter (perte d'information critique).\n")
    print("\n")
    print(" VARIABLES CATÉGORIELLES :")
    print(
        "   1-Catégorie  'Inconnu' (Recommandé en Scoring) : En banque/crédit,"
    )
    print("l'absence de donnée est un signal métier en soi. Par exemple,  \n")
    print(
        "l'absence d'information sur le statut de logement peut indiquer un"
        " risque plus élevé.\n"
    )
    print("   2-Mode : À utiliser si l'absence est un simple oubli technique")
    print("     sur une variable très homogène.")
    print(
        "   3-Supprimer : À éviter sauf si le volume de NaN est négligeable"
        " (< 1%)."
    )
    print("\n")


###############################################################""


def demander_choix():
    print("Variables numériques avec valeurs manquantes :")
    print("Veillez choisir  une option :")
    print("1. Remplacer par la médiane")
    print("2. Remplacer par la moyenne")
    print("3. Supprimer les lignes")

    choix_num = input("Entrez votre choix pour  ces valeurs numériques(1, 2 ou 3) : ")
    while choix_num not in ["1", "2", "3"]:
        print("Choix invalide. Veuillez choisir 1, 2 ou 3.")

        choix_num = input("Entrez votre choix pour  ces valeurs numériques(1, 2 ou 3) : ")

    print("Variables catégorielles avec valeurs manquantes :")
    print("Veillez choisir  une option :")
    print("1. Remplacer par la catégorie 'Inconnu'")
    print("2. Remplacer par le mode")
    print("3. Supprimer les lignes")

    choix_cat = input("Entrez votre choix pour  ces valeurs categorielles : (1, 2 ou 3) : ")

    while choix_cat not in ["1", "2", "3"]:
        print("Choix invalide. Veuillez choisir 1, 2 ou 3.")
        choix_cat = input("Entrez votre choix pour  ces valeurs categorielles : (1, 2 ou 3) : ")

    return choix_num, choix_cat


############################################################


def traiter_valeurs_manquantes(data_clean, choix_num, choix_cat):
    # traitement des variables numériques
    if data_clean.isnull().sum().sum() == 0:
        print("Aucune valeur manquante detectée.")
        return data_clean, None
    rapport = {
        "choix_numerique": choix_num,
        "choix_categoriel": choix_cat,
        "nb_lignes_avant_traitement": data_clean.shape[0],
        "nb_colonnes_avant_traitement": data_clean.shape[1],
        "nb_lignes_apres_traitement": None,  # Sera mis à jour à la fin,
        "nb_valeurs_manquantes_avant": data_clean.isnull().sum().sum(),
        "nb_valeurs_manquantes_apres": None,
             }
    col_na = data_clean.columns[data_clean.isnull().any()].tolist()
    col_na_num = data_clean[col_na].select_dtypes(include=[np.number]).columns
    col_na_cat = data_clean[col_na].select_dtypes(include=["object", "category"] ).columns



    if choix_num == "1":  # mediane
        for col in col_na_num:
            mediane = data_clean[col].median()
            data_clean[col] = data_clean[col].fillna(mediane)
    elif choix_num == "2":  # moyenne
        for col in col_na_num:
            moyenne = data_clean[col].mean()
            data_clean[col] = data_clean[col].fillna(moyenne)
    elif choix_num == "3":  # supprimer les lignes
        data_clean = data_clean.dropna(subset=col_na_num)

    # Traitement des variables catégorielles

    if choix_cat == "1":  # categorie 'Inconnu'
        for col in col_na_cat:
            if hasattr(data_clean[col], "cat") and (
                "Inconnu" not in data_clean[col].cat.categories
            ):
                data_clean[col] = data_clean[col].cat.add_categories("Inconnu")
            data_clean[col] = data_clean[col].fillna("Inconnu")
    elif choix_cat == "2":  # mode
        for col in col_na_cat:
            mode = data_clean[col].mode()[0]
            data_clean[col] = data_clean[col].fillna(mode)
    elif choix_cat == "3":  # supprimer les lignes
        data_clean = data_clean.dropna(subset=col_na_cat)

    rapport["nb_lignes_apres_traitement"] = data_clean.shape[0]
    rapport["nb_valeurs_manquantes_apres"] = data_clean.isnull().sum().sum()
    return data_clean, rapport

In [15]:

afficher_regles_expert()
choix_num, choix_cat = demander_choix()
df_clean, rapport = traiter_valeurs_manquantes(data, choix_num, choix_cat)

Voici les recommandations des experts data scientists / MLOps pour le traitement des valeurs manquantes :
1. VARIABLES NUMÉRIQUES :


   1- Médiane (Recommandé) : Indispensable en cas de valeurs extrêmes
     (outliers) pour ne pas déformer la distribution.
   2- Moyenne : À réserver aux distributions parfaitement normales.
   3-Supprimer : À éviter (perte d'information critique).



 VARIABLES CATÉGORIELLES :
   1-Catégorie  'Inconnu' (Recommandé en Scoring) : En banque/crédit,
l'absence de donnée est un signal métier en soi. Par exemple,  

l'absence d'information sur le statut de logement peut indiquer un risque plus élevé.

   2-Mode : À utiliser si l'absence est un simple oubli technique
     sur une variable très homogène.
   3-Supprimer : À éviter sauf si le volume de NaN est négligeable (< 1%).


Variables numériques avec valeurs manquantes :
Veillez choisir  une option :
1. Remplacer par la médiane
2. Remplacer par la moyenne
3. Supprimer les lignes
Variables catégorielles ave

In [ ]:
import numpy as np


def supprimer_valeurs_aberrantes(data_clean):

    # ============================================================
    # 1. Sélectionner uniquement les variables numériques
    # ============================================================

    colonnes_numeriques = data_clean.select_dtypes(
        include=[np.number]
    ).columns.tolist()


   

    if "statut_pret" in colonnes_numeriques:
        colonnes_numeriques.remove("statut_pret")


   

    for colonne in colonnes_numeriques:

        # Premier quartile
        premier_quartile = data_clean[colonne].quantile(0.25)

        # Troisième quartile
        troisieme_quartile = data_clean[colonne].quantile(0.75)

        # Écart interquartile
        ecart_interquartile = (
            troisieme_quartile - premier_quartile
        )

       
        
        borne_inferieure = (
            premier_quartile
            - 1.5 * ecart_interquartile
        )

        borne_superieure = (
            troisieme_quartile
            + 1.5 * ecart_interquartile
        )

        #
        # 5. Supprimer les lignes contenant des valeurs
        #    aberrantes pour cette variable
       

        data_clean = data_clean[
            (data_clean[colonne] >= borne_inferieure)
            &
            (data_clean[colonne] <= borne_superieure)
        ]


    # 6. Réinitialiser les index
   

    data_clean = data_clean.reset_index(drop=True)

    return data_clean



In [27]:
df=  supprimer_valeurs_aberrantes(df_clean)

In [28]:
# Corrections des  types de données pour les colonnes catégorielles

colonnes_categorielles = [
    "statut_logement",
    "motif_pret",
    "niveau_de_risque",
    "defaut_historique_credit"
]

for colonne in colonnes_categorielles:
    data_clean[colonne] = data_clean[colonne].astype("category")

print(data_clean.dtypes)

age                            int64
revenu_annuel                  int64
statut_logement             category
anciennete_emploi            float64
motif_pret                  category
niveau_de_risque            category
montant_pret                   int64
taux_interet                 float64
statut_pret                    int64
ratio_pret_revenu            float64
defaut_historique_credit    category
duree_historique_credit        int64
dtype: object


In [ ]:
#Séparation des données en variables explicatives (X) et variable cible (y)

X = df.drop("statut_pret", axis=1)
Y = df["statut_pret"]




statut_pret
0    20852
1     5297
Name: count, dtype: int64


In [ ]:
df.to_csv("../data/data_clean.csv", index=False)

: 